In [ ]:
!pip install spotipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 8.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import os
import time
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from google.colab import userdata, drive

# 1. Preparação do Ambiente
drive.mount('/content/drive')
caminho_pasta = '/content/drive/MyDrive/Projetos/RecomendacaoSpotify'
arquivo_saida = f'{caminho_pasta}/metadata_enriquecido.csv'


# if os.path.exists('.cache'):
#     os.remove('.cache')

# 1. Autenticação
client_id = userdata.get('SPOTIFY_CLIENT_ID')
client_secret = userdata.get('SPOTIFY_CLIENT_SECRET')

auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)

# Autenticação e conexão
sp = spotipy.Spotify(auth_manager=auth_manager)
print("\nAutenticado com sucesso no Spotify!")

# 2. Carregar a lista de "busca" e o progresso atual
df_elite = pd.read_csv(f'{caminho_pasta}/uris_unicas5.csv')

if os.path.exists(arquivo_saida):
    df_progresso = pd.read_csv(arquivo_saida)
    ids_processados = set(df_progresso['spotify_track_uri'].unique())
    print(f"🔄 Retomando: {len(ids_processados)} obras já encontradas.")
else:
    ids_processados = set()
    print("🆕 Começando nova coleta.")

# Filtrar o que ainda falta buscar
obras_para_buscar = df_elite[~df_elite['spotify_track_uri'].isin(ids_processados)]
print(f"🚀 Faltam {len(obras_para_buscar)} obras para processar.")

# 3. Função de Coleta com Checkpoint
def coletar_metadados(df_alvo):
    # 1. Calcula o tamanho do passo (5%) antes de começar o loop
    total_lote = len(df_alvo)
    passo_5_percent = max(1, total_lote // 20) # Garante que o passo seja pelo menos 1 para lotes pequenos (5% de 20 é 1)

    print(f"📊 Iniciando coleta de {total_lote} registros. Avisos a cada {passo_5_percent} itens (5% de progresso).")

    pause_counter = 0 # Initialize counter for the 50-track pause

    for i, row in df_alvo.reset_index(drop=True).iterrows():
        track_id = row['spotify_track_uri']
        try:
            # Busca unitária (já que o lote deu 403)
            track_info = sp.track(track_id)

            # Organiza os dados
            dados = {
                'spotify_track_uri': track_id,
                'popularidade': track_info.get('popularity', 0),
                'duracao_ms': track_info.get('duration_ms'),
                'explicit': track_info.get('explicit'),
                'data_release': track_info['album'].get('release_date')
            }

            # Transforma em DataFrame de uma linha
            df_temp = pd.DataFrame([dados])

            # SALVAMENTO IMEDIATO (Append mode)
            # Se o arquivo não existe, cria com cabeçalho. Se existe, apenas anexa.
            df_temp.to_csv(arquivo_saida, mode='a', index=False, header=not os.path.exists(arquivo_saida))

            posicao = i + 1
            if posicao % passo_5_percent == 0 or posicao == total_lote:
                percentual = (posicao / total_lote) * 100
                print(f"✅ {percentual:.0f}% concluído ({posicao} / {total_lote})")


            # Delay preventivo para o Rate Limit (per track)
            time.sleep(2.0)

        except Exception as e:
            if "429" in str(e):
                print("🛑 Rate Limit atingido! O Spotify pediu para parar. Pausando e aguardando...")
                time.sleep(120) # Wait for 2 minutes if rate limit is hit
                print("▶️ Tentando novamente após a pausa do Rate Limit.")
                continue # Continue to the next track after the forced pause.

            print(f"⚠️ Erro no ID {track_id}: {e}")


# 4. Pegar apenas as próximas músicas da fila de espera
qt_por_lote = 200

lote_do_dia = obras_para_buscar.head(qt_por_lote)

print(f"📋 Total pendente: {len(obras_para_buscar)}")
print(f"🚀 Iniciando processamento de apenas {len(lote_do_dia)} registros para evitar bloqueios.")

# 5. Chamar a função de coleta passando apenas essa fatia
coletar_metadados(lote_do_dia)

print(f"\n🏁 Lote de {qt_por_lote} finalizado. O arquivo no Drive foi atualizado.")

Mounted at /content/drive

Autenticado com sucesso no Spotify!
🔄 Retomando: 6181 obras já encontradas.
🚀 Faltam 4 obras para processar.
📋 Total pendente: 4
🚀 Iniciando processamento de apenas 4 registros para evitar bloqueios.
📊 Iniciando coleta de 4 registros. Avisos a cada 1 itens (5% de progresso).
✅ 25% concluído (1 / 4)
✅ 50% concluído (2 / 4)
✅ 75% concluído (3 / 4)
✅ 100% concluído (4 / 4)

🏁 Lote de 200 finalizado. O arquivo no Drive foi atualizado.


In [ ]:
df_final = pd.read_csv(arquivo_saida)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6185 entries, 0 to 6184
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   spotify_track_uri  6185 non-null   object
 1   popularidade       6185 non-null   int64 
 2   duracao_ms         6185 non-null   int64 
 3   explicit           6185 non-null   bool  
 4   data_release       6185 non-null   object
dtypes: bool(1), int64(2), object(2)
memory usage: 199.5+ KB


In [ ]:
df_final.head(20)

,spotify_track_uri,popularidade,duracao_ms,explicit,data_release
0,spotify:track:5wfd0Uu6qP19vCCTn5le2e,0,251613,False,2004-02-17
1,spotify:track:5AVGO9BDNucuEb80GKmc5N,0,233933,False,1982-07-20
2,spotify:track:7aDnfV0hfqwneOzNjcymii,0,203506,False,1981-09-02
3,spotify:track:6n52cvPib8EfFfORi8kPVB,0,233093,False,1983-05-03
4,spotify:track:3NLrRZoMF0Lx6zTlYqeIo4,0,238733,False,2002-11-12
5,spotify:track:3DelzIn0lPwX4PeOJk9zDF,0,241960,False,2008-01-01
6,spotify:track:6ZOBP3NvffbU4SZcrnt1k6,0,233933,False,2000
7,spotify:track:3WbphvawbMZ8FyqDxYGdSQ,0,260333,False,2002-11-12
8,spotify:track:6LgjN2jxrDE77W7vMjeW7S,0,227294,False,2019-05-10
9,spotify:track:70oUkZVnH2WQWaNZ8zwn8p,0,227294,False,2019-05-10


# Testes Kaggle

In [ ]:
# ============================================================
# CRUZAMENTO COM KAGGLE - Spotify Tracks Dataset
# Rode no Google Colab
# ============================================================
# Pré-requisito: ter o kaggle.json configurado no Colab
# ou baixar manualmente o dataset em:
# https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset
# e fazer upload do arquivo 'dataset.csv' no Colab
# ============================================================

import pandas as pd
import os

# ------------------------------------------------------------
# OPÇÃO A: Baixar automaticamente via Kaggle API
# (precisa do kaggle.json configurado)
# ------------------------------------------------------------
# from google.colab import files
# files.upload()  # suba o kaggle.json aqui
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d maharshipandya/-spotify-tracks-dataset --unzip

# ------------------------------------------------------------
# OPÇÃO B: Baixar manualmente e fazer upload no Colab
# Link: https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset
# Arquivo esperado: dataset.csv
# ------------------------------------------------------------

# ============================================================
# 1. Carregar suas URIs
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

caminho_pasta = '/content/drive/MyDrive/Projetos/RecomendacaoSpotify'

df_uris = pd.read_csv(f'{caminho_pasta}/uris_unicas.csv')
df_uris['track_id'] = df_uris['spotify_track_uri'].str.replace('spotify:track:', '', regex=False)
print(f"✅ Suas URIs carregadas: {len(df_uris)} faixas")

# ============================================================
# 2. Carregar o dataset do Kaggle
# ============================================================
# Ajuste o caminho se necessário
df_kaggle = pd.read_csv(f'{caminho_pasta}/dataset.csv')
print(f"✅ Dataset Kaggle carregado: {len(df_kaggle)} faixas")
print(f"   Colunas disponíveis: {df_kaggle.columns.tolist()}")

# ============================================================
# 3. Cruzamento pelo track_id
# ============================================================
# O dataset do Kaggle usa a coluna 'track_id'
df_cruzado = df_uris.merge(
    df_kaggle[['track_id', 'duration_ms']],
    on='track_id',
    how='left'
)

# ============================================================
# 4. Separar cobertas vs. faltantes
# ============================================================
df_cobertas = df_cruzado[df_cruzado['duration_ms'].notna()].copy()
df_faltantes = df_cruzado[df_cruzado['duration_ms'].isna()].copy()

total = len(df_uris)
cobertas = len(df_cobertas)
faltantes = len(df_faltantes)

print(f"\n📊 RESULTADO DO CRUZAMENTO:")
print(f"   Total de faixas:         {total}")
print(f"   ✅ Encontradas no Kaggle: {cobertas} ({cobertas/total*100:.1f}%)")
print(f"   ❌ Precisam de API:       {faltantes} ({faltantes/total*100:.1f}%)")
print(f"   💰 Chamadas economizadas: {cobertas}")

# ============================================================
# 5. Salvar os resultados
# ============================================================

# Faixas já resolvidas pelo Kaggle
df_cobertas.to_csv(f'{caminho_pasta}/metadata_kaggle.csv', index=False)
print(f"\n💾 Salvo: metadata_kaggle.csv ({cobertas} faixas com duration_ms e release_date)")

# Apenas as URIs que ainda precisam de chamada de API
df_faltantes[['artista', 'musica', 'spotify_track_uri']].to_csv(
    f'{caminho_pasta}/uris_pendentes_api.csv', index=False
)
print(f"💾 Salvo: uris_pendentes_api.csv ({faltantes} faixas ainda precisam de API)")

print("\n🏁 Pronto! Use o uris_pendentes_api.csv no seu script de coleta.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Suas URIs carregadas: 9289 faixas
✅ Dataset Kaggle carregado: 114000 faixas
   Colunas disponíveis: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

📊 RESULTADO DO CRUZAMENTO:
   Total de faixas:         9289
   ✅ Encontradas no Kaggle: 2363 (25.4%)
   ❌ Precisam de API:       7854 (84.6%)
   💰 Chamadas economizadas: 2363

💾 Salvo: metadata_kaggle.csv (2363 faixas com duration_ms e release_date)
💾 Salvo: uris_pendentes_api.csv (7854 faixas ainda precisam de API)

🏁 Pronto! Use o uris_pendentes_api.csv no seu script de coleta.


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amitanshjoshi/spotify-1million-tracks")

print("Path to dataset files:", path)






100%|██████████| 77.1M/77.1M [00:00<00:00, 144MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/amitanshjoshi/spotify-1million-tracks/versions/1


In [ ]:
# 2. Ver o nome exato do arquivo
import os
print(os.listdir(path))

['spotify_data.csv']


In [ ]:
import kagglehub
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
caminho_pasta = '/content/drive/MyDrive/Projetos/RecomendacaoSpotify'

# 1. Baixar o dataset
path = kagglehub.dataset_download("amitanshjoshi/spotify-1million-tracks")

# 2. Carregar
df_uris = pd.read_csv(f'{caminho_pasta}/uris_unicas.csv')
df_uris['track_id'] = df_uris['spotify_track_uri'].str.replace('spotify:track:', '', regex=False)

df_kaggle = pd.read_csv(f'{path}/spotify_data.csv')
print(f"Colunas: {df_kaggle.columns.tolist()}")

# 3. Cruzamento
df_cruzado = df_uris.merge(
    df_kaggle[['track_id', 'duration_ms']],
    on='track_id',
    how='left'
)

# 4. Resultado
df_cobertas = df_cruzado[df_cruzado['duration_ms'].notna()]
df_faltantes = df_cruzado[df_cruzado['duration_ms'].isna()]

total = len(df_uris)
print(f"\n📊 RESULTADO:")
print(f"✅ Encontradas: {len(df_cobertas)} ({len(df_cobertas)/total*100:.1f}%)")
print(f"❌ Precisam de API: {len(df_faltantes)} ({len(df_faltantes)/total*100:.1f}%)")

# 5. Salvar
df_cobertas.to_csv(f'{caminho_pasta}/metadata_kaggle.csv', index=False)
df_faltantes[['artista', 'musica', 'spotify_track_uri']].to_csv(
    f'{caminho_pasta}/uris_pendentes_api.csv', index=False
)
print("💾 Arquivos salvos!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'spotify-1million-tracks' dataset.
Colunas: ['Unnamed: 0', 'artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature']

📊 RESULTADO:
✅ Encontradas: 995 (10.7%)
❌ Precisam de API: 8294 (89.3%)
💾 Arquivos salvos!
